In [1]:
import numpy as np
from tenpy.models.model import CouplingMPOModel, NearestNeighborModel
from tenpy.models.lattice import Chain
from tenpy.networks.site import SpinHalfSite, SpinHalfFermionSite, set_common_charges, GroupedSite
from tenpy.networks.mps import MPS
from tenpy.algorithms import dmrg, tebd
from tenpy.networks.mpo import MPO


In [ ]:
class KondoModel(CouplingMPOModel, NearestNeighborModel):
    r"""1D Kondo lattice model with conduction electrons and localized spins.
    
    H = -t \sum_{<ij>\sigma} c^\dagger_{i\sigma} c_{j\sigma} + h.c.
        + J_1 \sum_{i \sigma\sigma'} c^\dagger_{i\sigma} \vec{S}_{\sigma\sigma'} c_{i\sigma'} \cdot \vec{s}_i
        + J_2 \sum_{i} \vec{s}_i \cdot \vec{s}_{i+1}
    """

    def init_sites(self):
        ferm_site = SpinHalfFermionSite(cons_N='N', cons_Sz= None )
        spin_site = SpinHalfSite(conserve= None)
        set_common_charges([ferm_site, spin_site], new_charges='independent')
        
        site = GroupedSite([ferm_site, spin_site], labels = ['e', 'i'] , charges = 'independent')
        return site
   
    def init_lattice(self, model_params):
        L = model_params['L']
        bc = model_params.get("bc", "open")
        site = self.init_sites()
        lat = Chain(L, site, bc=bc, bc_MPS='finite')
        return lat
    
    def init_terms(self, model_params):
        t = model_params.get('t', 1.0)
        J1 = model_params.get('J1', 1.0)
        J2 = model_params.get('J2', 0.5)
        lat = self.init_lattice(model_params)
        
        # Electron hopping
        for u1, u2, dx in lat.pairs['nearest_neighbors']:
            self.add_coupling(-1.0 * t, u1, 'Cdue', u2, 'Cue', dx, plus_hc=True)
            self.add_coupling(-1.0 * t, u1, 'Cdde', u2, 'Cde', dx, plus_hc=True)
        # J1: Kondo coupling on each site
        # S_electron · s_impurity = S^z_e S^z_i + (S^+_e S^-_i + S^-_e S^+_i)/2
        for u in range(len(lat.unit_cell)):
            # Sz Sz coupling
            self.add_onsite_term(4.0 * J1, u, "Szi Sze")
            # S+ S- coupling
            self.add_onsite_term(2.0 * J1, u, "Spi Sme", plus_hc = False)  
            self.add_onsite_term(2.0 * J1, u, "Smi Spe", plus_hc = False)     
        
        # J2: Impurity-impurity Heisenberg coupling
        for u1, u2, dx in lat.pairs['nearest_neighbors']:
            self.add_coupling(4.0 * J2, u1, 'Szi', u2, 'Szi', dx)
            self.add_coupling(2.0 * J2, u1, 'Spi', u2, 'Smi', dx)
            self.add_coupling(2.0 * J2, u1, 'Smi', u2, 'Spi', dx)

    def build_Q_MPO(self, model_params, Q_op = 'Szi_tot'):
        """
        builds an MPO corresponding to Q
        """
        L = model_params['L']
        lat = self.init_lattice(model_params)

        Id = lat.unit_cell[0].get_op("Id")
        if Q_op == 'Szi_tot':
            Q = lat.unit_cell[0].get_op("Sigmazi")
        else:
            raise ValueError("Must pick a preconfigured operator")
        
        W = []

        for i in range(L):
            Wi = np.empty((2, 2), dtype=object)

            Wi[0, 0] = Id
            Wi[1, 1] = Id

            Wi[0, 1] = Q / L
            Wi[1, 0] = None

            W.append(Wi)
        return MPO.from_grids(lat.mps_sites(), W, IdL=0, IdR=1)

def run_dmrg(model_params, dmrg_params):
    """Run DMRG to find ground state."""
    model = KondoModel(model_params)
    lat = model.init_lattice(model_params)
        
    # Initial product state: empty conduction band, Neel impurity spins
    L = model_params['L']
    initial_state = []
    odd_sites_initial = dmrg_params.get("odd_sites_initial", 'empty_e down_i')
    even_sites_initial = dmrg_params.get("even_sites_initial", 'full_e down_i')
    for i in range(L):
        if i % 2 == 0:
            initial_state.append(even_sites_initial)  # electron site, impurity site
        else:
            initial_state.append(odd_sites_initial)
    #print(model.init_sites(model_params).state_labels)     #for fiding the names of the initial states
    psi = MPS.from_product_state(lat.mps_sites(), initial_state, bc=lat.bc_MPS)
    
    # DMRG engine
    engine = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
    E, psi = engine.run()
    
    print(f"\nDMRG Ground state energy: {E:.8f}")
    return model, psi, E


def run_tebd(model_params, tebd_params, initial_psi=None):
    """Run TEBD time evolution."""
    model = KondoModel(model_params)
    model.calc_H_bond()
    
    if initial_psi is None:
        # Create initial state
        L = model_params['L']
        initial_state = []
        for i in range(L):
            if i % 2 == 0:
                initial_state.append('full_e down_i')
            else:
                initial_state.append('empty_e up_i')
        psi = MPS.from_product_state(model.lat.mps_sites(), initial_state, bc=model.lat.bc_MPS)
    else:
        psi = initial_psi.copy()
    
    # TEBD engine
    engine = tebd.TEBDEngine(psi, model, tebd_params)
    
    times = []
    energies = []
    measurements = []
    
    for step in range(tebd_params.get('N_steps', 10)):
        engine.run()
        times.append(engine.evolved_time)
        energies.append(model.H_MPO.expectation_value(psi))
        
        # Measure local observables
        L = model_params['L']
        Sz_e = psi.expectation_value('Sze')
        Sz_i = psi.expectation_value('Szi')
        
        measurements.append({
            'time': engine.evolved_time,
            'energy': energies[-1],
            'Sz_e': Sz_e.copy(),
            'Sz_i': Sz_i.copy()
        })
        
        verbosity = tebd_params.get('verbosity', 0)
        if (verbosity > 0) or step % max(1, tebd_params['N_steps'] // 10) == 0:
            print(f"Time step {step+1}/{tebd_params['N_steps']}: "
                  f"t={engine.evolved_time:.4f}, E={energies[-1]:.8f}")
    
    return psi, times, energies, measurements

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [3]:
model_params = {"t": 1, "J1" : 1, "J2" : 2, "L" : 4, "bc" : "open"}
dmrg_params = {'mixer': True,'mixer_params': {'amplitude': 1e-5, 'decay': 2.0, 'disable_after': 30,        },
        'trunc_params': {'chi_max': 20, 'svd_min': 1e-10, 'trunc_cut': 1e-6,},
        'max_sweeps': 10, 'max_E_err': 1e-8, 'verbose': 1,
        "odd_sites_initial" : 'empty_e down_i' , "even_sites_initial": 'full_e down_i'}
tebd_params = {'dt': 0.05,'N_steps': 2, 
               'trunc_params': {'chi_max': 20, 'svd_min': 1e-10, 'trunc_cut': 1e-6, },
        'order': 2, 'verbosity': 1}

model, psi, E0 = run_dmrg(model_params, dmrg_params)
model.build_Q_MPO(model_params)



c:\Users\ianro\Documents\Research\Quantum_Metrology_AKLT\.venv\Lib\site-packages\tenpy\networks\mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.75e-05



DMRG Ground state energy: -16.63793332


c:\Users\ianro\Documents\Research\Quantum_Metrology_AKLT\.venv\Lib\site-packages\tenpy\tools\params.py:243: UserWarning: unused options for config TwoSiteDMRGEngine:
['even_sites_initial', 'odd_sites_initial', 'verbose']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
c:\Users\ianro\Documents\Research\Quantum_Metrology_AKLT\.venv\Lib\site-packages\tenpy\networks\mpo.py:151: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width=mps_unit_cell_width)


In [11]:
class AndersonImpurityModel(CouplingMPOModel):

    def init_sites(self, model_params):
        conserve = model_params.get('conserve', 'N')
        return SpinHalfFermionSite(cons_N=conserve)

    def init_lattice(self, model_params):
        L = model_params['L']
        site = self.init_sites(model_params)
        return Chain(L, site, bc='open')

    def init_terms(self, model_params):

        t = model_params['t']
        U = model_params['U']
        eps_d = model_params['eps_d']

        L = self.lat.N_sites
        imp = L // 2

        # nearest-neighbor hopping
        for i in range(L - 1):

            self.add_coupling(
                -t, i, 'Cdu', i + 1, 'Cu', dx = 1, 
                plus_hc=True
            )

            self.add_coupling(
                -t, i, 'Cdd', i + 1, 'Cd', dx = 1, 
                plus_hc=True
            )

        # impurity onsite energy
        self.add_onsite(eps_d, imp, 'Ntot')

        # impurity Hubbard interaction
        self.add_onsite(U, imp, 'NuNd')

In [12]:
model_params = {
    'L': 40,
    't': 1.0,
    'U': 8.0,
    'eps_d': -4.0,      # particle-hole symmetric point
    'conserve': 'N'
}

model = AndersonImpurityModel(model_params)

IndexError: list index out of range